# CosMx Human Multiomic Breast — assemble AnnData

Builds a single `.h5ad` file from the Bruker CosMx multiomic breast flat files (RNA + Protein, same slide).
Output is structured for KaroSpace:
- `X` / `layers['counts']` — RNA counts (cells × genes)
- `obs.x`, `obs.y`, `obsm['spatial']` — global µm coords
- `obsm['protein']` + `uns['protein_var']` — 64-plex protein per-cell intensities
- `obs.sample`, `obs.fov`, `obs.cell` — identifiers

In [1]:
import gzip, io, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

ROOT  = Path('/Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast')
OUT_H5AD = ROOT/'cosmx_multiomic_breast.h5ad'
RNA_ZIP  = ROOT/'Flatfiles_RNA.zip'
PRO_ZIP  = ROOT/'Flatfiles_Protein.zip'
EX_DIR   = ROOT/'extracted'
EX_DIR.mkdir(exist_ok=True)
PX_TO_UM = 0.120  # CosMx: ~120 nm/px (verified from fov_positions: px ↔ mm ratio)
SAMPLE_NAME = 'BreastCancer_multiomic'

In [2]:
# Extract zips if not already done
def ensure_extracted(zip_path, marker_glob):
    target = EX_DIR/zip_path.stem
    if list(target.rglob(marker_glob)):
        print('already extracted:', target)
        return target
    print('unzipping', zip_path, '→', target)
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target)
    return target

rna_root = ensure_extracted(RNA_ZIP, '*exprMat_file.csv.gz')
pro_root = ensure_extracted(PRO_ZIP, '*exprMat_file.csv.gz')

def find_one(root, suffix):
    hits = list(root.rglob(f'*{suffix}'))
    assert len(hits)==1, f'expected exactly 1 match for {suffix} under {root}, got {hits}'
    return hits[0]

rna_files = {k: find_one(rna_root, v) for k,v in {
    'expr':      'exprMat_file.csv.gz',
    'meta':      'metadata_file.csv.gz',
    'fov':       'fov_positions_file.csv.gz',
    'polygons':  'polygons.csv.gz',
}.items()}
pro_files = {k: find_one(pro_root, v) for k,v in {
    'expr':      'exprMat_file.csv.gz',
    'meta':      'metadata_file.csv.gz',
    'fov':       'fov_positions_file.csv.gz',
    'polygons':  'polygons.csv.gz',
}.items()}
print('RNA files:', rna_files)
print('Protein files:', pro_files)

unzipping /Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast/Flatfiles_RNA.zip → /Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast/extracted/Flatfiles_RNA
unzipping /Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast/Flatfiles_Protein.zip → /Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast/extracted/Flatfiles_Protein
RNA files: {'expr': PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast/extracted/Flatfiles_RNA/Flatfiles_RNA/flatFiles/BreastCancer/BreastCancer_exprMat_file.csv.gz'), 'meta': PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling/notebooks/data/cosmx_multiomic_breast/extracted/Flatfiles_RNA/Flatf

In [ ]:
# --- Load RNA expression matrix (cells × genes) ---
# Header: fov, cell_ID, gene1, gene2, ..., NegPrb*, SystemControl*
print('reading RNA exprMat (large) ...')
rna_expr = pd.read_csv(rna_files['expr'])
print(rna_expr.shape, 'columns[:6]=', list(rna_expr.columns[:6]))
rna_expr['cell_uid'] = rna_expr['fov'].astype(str) + '_' + rna_expr['cell_ID'].astype(str)
feat_cols = [c for c in rna_expr.columns if c not in ('fov','cell_ID','cell_uid')]
is_neg = np.array([c.lower().startswith(('negprb','negprobe','neg_prb','negative')) for c in feat_cols])
is_sys = np.array([c.lower().startswith(('systemcontrol','sys_control','falsecode')) for c in feat_cols])
is_real = ~(is_neg | is_sys)
real_genes = [g for g,r in zip(feat_cols, is_real) if r]
neg_genes  = [g for g,r in zip(feat_cols, is_neg)  if r]
sys_genes  = [g for g,r in zip(feat_cols, is_sys)  if r]
print(f'real genes: {len(real_genes)} | neg probes: {len(neg_genes)} | sys controls: {len(sys_genes)}')

reading RNA exprMat (large) ...


In [ ]:
# Build sparse RNA count matrix (cells × genes) and per-cell QC for neg/sys
X_rna = sp.csr_matrix(rna_expr[real_genes].to_numpy(dtype=np.int32))
neg_total = rna_expr[neg_genes].sum(axis=1).to_numpy() if neg_genes else np.zeros(len(rna_expr))
sys_total = rna_expr[sys_genes].sum(axis=1).to_numpy() if sys_genes else np.zeros(len(rna_expr))
rna_obs = pd.DataFrame({
    'fov':      rna_expr['fov'].astype(int).values,
    'cell_ID':  rna_expr['cell_ID'].astype(int).values,
    'cell_uid': rna_expr['cell_uid'].values,
    'nCount_neg': neg_total,
    'nCount_sys': sys_total,
})
print('RNA matrix:', X_rna.shape, 'nnz=', X_rna.nnz)
del rna_expr

In [ ]:
# --- Cell metadata (RNA flat files) ---
rna_meta = pd.read_csv(rna_files['meta'])
rna_meta['cell_uid'] = rna_meta['fov'].astype(str) + '_' + rna_meta['cell_ID'].astype(str)
print('rna_meta', rna_meta.shape, 'cols sample:', list(rna_meta.columns[:20]))
# Keep only the columns useful for KaroSpace + QC
keep_meta_cols = [c for c in [
    'cell_uid','fov','cell_ID','cell','CenterX_local_px','CenterY_local_px',
    'CenterX_global_px','CenterY_global_px','Area','Area.um2','AspectRatio',
    'Width','Height','Circularity','Eccentricity','Solidity','Perimeter',
    'Mean.DAPI','Max.DAPI','Mean.PanCK','Max.PanCK','Mean.CD45','Max.CD45',
    'Mean.CD68','Max.CD68','Mean.Membrane','Max.Membrane',
    'nCount_RNA','nFeature_RNA','nCount_negprobes','nFeature_negprobes',
    'slide_ID','tissue','Panel','assay_type','Run_Tissue_name','Run_name'
] if c in rna_meta.columns]
rna_meta = rna_meta[keep_meta_cols].copy()
rna_obs = rna_obs.merge(rna_meta, on=['cell_uid','fov','cell_ID'], how='left')
print('rna_obs:', rna_obs.shape)

In [ ]:
# --- Protein expression matrix (cells × proteins, mean intensities) ---
pro_expr = pd.read_csv(pro_files['expr'])
pro_expr['cell_uid'] = pro_expr['fov'].astype(str) + '_' + pro_expr['cell_ID'].astype(str)
pro_feat_cols = [c for c in pro_expr.columns if c not in ('fov','cell_ID','cell_uid')]
iso_mask = np.array([c.startswith(('Ms IgG','Rb IgG')) for c in pro_feat_cols])
channel_mask = np.array([c.startswith('Channel-') for c in pro_feat_cols])
real_pro = [p for p,iso,ch in zip(pro_feat_cols, iso_mask, channel_mask) if not (iso or ch)]
iso_pro  = [p for p,iso in zip(pro_feat_cols, iso_mask) if iso]
ch_pro   = [p for p,ch  in zip(pro_feat_cols, channel_mask) if ch]
print(f'protein features: real={len(real_pro)} isotypes={len(iso_pro)} channels={len(ch_pro)}')

In [ ]:
# --- Align cells between RNA and Protein by (fov, cell_ID) ---
rna_idx = pd.Index(rna_obs['cell_uid'])
pro_idx = pd.Index(pro_expr['cell_uid'])
shared  = rna_idx.intersection(pro_idx)
print(f'cells: RNA={len(rna_idx)}  Protein={len(pro_idx)}  shared={len(shared)}')

# Subset and reorder both to shared cells, RNA order = canonical
keep = rna_obs['cell_uid'].isin(shared).values
X_rna  = X_rna[keep]
rna_obs = rna_obs.loc[keep].reset_index(drop=True)
pro_expr = pro_expr.set_index('cell_uid').loc[rna_obs['cell_uid'].values].reset_index()
assert (pro_expr['cell_uid'].values == rna_obs['cell_uid'].values).all()
print('aligned cells:', X_rna.shape[0])

P_real = pro_expr[real_pro].to_numpy(dtype=np.float32)
P_iso  = pro_expr[iso_pro].to_numpy(dtype=np.float32) if iso_pro else None
P_ch   = pro_expr[ch_pro].to_numpy(dtype=np.float32)  if ch_pro  else None
print('protein matrix:', P_real.shape)

In [ ]:
# --- Spatial coords (µm) from RNA metadata ---
rna_obs['x'] = rna_obs['CenterX_global_px'].astype(float) * PX_TO_UM
rna_obs['y'] = rna_obs['CenterY_global_px'].astype(float) * PX_TO_UM
rna_obs['sample'] = SAMPLE_NAME
rna_obs.index = rna_obs['cell_uid'].astype(str)
rna_obs.index.name = 'obs'
rna_obs.head(3)

In [ ]:
# --- Build AnnData ---
var_df = pd.DataFrame(index=pd.Index(real_genes, name='gene'))
var_df['feature_type'] = 'gene'

adata = ad.AnnData(
    X=X_rna,
    obs=rna_obs.drop(columns=['cell_uid']),
    var=var_df,
    layers={'counts': X_rna},
    obsm={'spatial': rna_obs[['x','y']].to_numpy(dtype=np.float32)},
)

# Protein modality: store as obsm + var-level metadata in uns
adata.obsm['protein'] = P_real
adata.uns['protein_var'] = pd.DataFrame({'protein': real_pro, 'feature_type': 'protein'})
if P_iso is not None:
    adata.obsm['protein_isotype'] = P_iso
    adata.uns['protein_isotype_var'] = pd.DataFrame({'protein': iso_pro, 'feature_type': 'isotype'})
if P_ch is not None:
    adata.obsm['protein_channel'] = P_ch
    adata.uns['protein_channel_var'] = pd.DataFrame({'protein': ch_pro, 'feature_type': 'channel'})

# FOV positions table
fov_pos = pd.read_csv(rna_files['fov'])
adata.uns['fov_positions'] = fov_pos

print(adata)

In [ ]:
# Quick sanity check
print('total RNA counts per cell median:', np.median(np.asarray(X_rna.sum(axis=1)).ravel()))
print('protein matrix median:', float(np.median(P_real)))
print('x range µm:', adata.obs['x'].min(), '..', adata.obs['x'].max())
print('y range µm:', adata.obs['y'].min(), '..', adata.obs['y'].max())

In [ ]:
# --- Save .h5ad ---
adata.write_h5ad(OUT_H5AD, compression='gzip')
print('wrote', OUT_H5AD, '·', round(OUT_H5AD.stat().st_size/1e6,1), 'MB')